## 1. Configuration

Modify these settings for your specific simulation.

In [ ]:
# ============================================================================
# USER CONFIGURATION
# ============================================================================

# Simulation type: 'PINACLES' or 'SCREAM'
SIMULATION_TYPE = 'PINACLES'  # Options: 'PINACLES', 'SCREAM'

# For PINACLES simulations with multiple part files
# Set to None if single file or SCREAM
PINACLES_FILE_PATTERN = '/pscratch/sd/p/paccini/temp/temp_imse_budget/ds_2d_600x600_3km_SCREAMinit_part*.nc'
PINACLES_NUM_PARTS = 9  # Number of part files (e.g., part1.nc through part9.nc)

# For SCREAM or single-file datasets
SCREAM_FILE_PATH = '/pscratch/sd/p/paccini/temp/output_dp_scream/OLR_600x600_regridded.nc'

# Variable names
OLR_VAR_NAME = 'toa_lw_up'  # For PINACLES: 'toa_lw_up', For SCREAM: 'OLR'

# Domain information
DOMAIN_SIZE = '600x600'  # e.g., '600x600', '500x500'
RESOLUTION = '3km'       # e.g., '3km', '1km'

# Processing parameters
OLR_THRESHOLD = 173      # W/m² - threshold for deep convection (OLR < threshold)
USE_PERIODIC = True      # Use periodic boundary conditions

# Output configuration
OUTPUT_DIR = f'index_pkl_{DOMAIN_SIZE}_{RESOLUTION}'
OUTPUT_FILENAME = f'df_{SIMULATION_TYPE}_{DOMAIN_SIZE}_{RESOLUTION}_periodic_hourly.pkl'

# Experiment label (for plot titles)
EXPERIMENT_LABEL = f'{SIMULATION_TYPE} {DOMAIN_SIZE} {RESOLUTION}'

print(f"Configuration loaded for: {EXPERIMENT_LABEL}")
print(f"Output will be saved to: {OUTPUT_DIR}/{OUTPUT_FILENAME}")

## 2. Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import matplotlib.dates as mdates

## 3. Import Organization Index Functions

In [ ]:
# Import fixed periodic implementation
from run_metrics_periodic_fixed import run_metrics_periodic
from run_metrics import run_metrics

print("✓ Fixed periodic boundary implementation loaded")

## 4. Load Data

Load OLR data based on simulation type.

In [ ]:
if SIMULATION_TYPE == 'PINACLES':
    print(f"Loading PINACLES data from {PINACLES_NUM_PARTS} part files...")
    
    # Load all part files
    ds_parts = []
    for i in range(1, PINACLES_NUM_PARTS + 1):
        file_path = PINACLES_FILE_PATTERN.replace('*', f'{i}')
        print(f"  Loading part {i}/{PINACLES_NUM_PARTS}...", end=' ')
        ds_parts.append(xr.open_dataset(file_path))
        print("✓")
    
    # Concatenate along time dimension
    ds = xr.concat(ds_parts, 'time')
    print(f"✓ Concatenated {PINACLES_NUM_PARTS} files")
    
elif SIMULATION_TYPE == 'SCREAM':
    print(f"Loading SCREAM data from: {SCREAM_FILE_PATH}")
    ds = xr.open_dataset(SCREAM_FILE_PATH)
    print("✓ Data loaded")

else:
    raise ValueError(f"Unknown SIMULATION_TYPE: {SIMULATION_TYPE}. Use 'PINACLES' or 'SCREAM'.")

# Extract OLR variable
olr_data = ds[OLR_VAR_NAME].copy()

print(f"\nData shape: {olr_data.shape}")
print(f"Time range: {olr_data.time.values[0]} to {olr_data.time.values[-1]}")
print(f"Number of time steps: {len(olr_data.time)}")

## 5. Visualize Sample Time Steps

In [ ]:
# Plot a few sample time steps to check data
n_samples = min(7, len(olr_data.time))
sample_indices = np.linspace(0, len(olr_data.time)-1, n_samples, dtype=int)

fig, axes = plt.subplots(1, n_samples, figsize=(3*n_samples, 3.5))
if n_samples == 1:
    axes = [axes]

for ax, idx in zip(axes, sample_indices):
    olr_data.isel(time=idx).plot(ax=ax, robust=True, cmap='gray_r', add_colorbar=True)
    ax.set_title(f'Time step {idx}', fontsize=10)
    ax.set_aspect('equal')

plt.suptitle(f'{EXPERIMENT_LABEL} - OLR Sample Time Steps', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Define Processing Function

In [ ]:
def process_organization_indices(ds, time_coord='time', cut=173, periodic=True):
    """
    Compute organization indices for each time step.
    
    Parameters:
    -----------
    ds : xarray.DataArray
        OLR data array with time dimension
    time_coord : str
        Name of time coordinate
    cut : float
        OLR threshold for deep convection (W/m²)
    periodic : bool
        Use periodic boundary conditions
    
    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with organization indices for each time step
    """
    df = pd.DataFrame()
    time_values = ds.coords[time_coord].values
    n_times = len(time_values)
    
    print(f"Processing {n_times} time steps...")
    
    for i, time_value in enumerate(time_values):
        if (i + 1) % 100 == 0 or i == 0:
            print(f"  Progress: {i+1}/{n_times} ({100*(i+1)/n_times:.1f}%)")
        
        ds_time = ds.sel({time_coord: time_value})
        
        # Extract date and time information
        date_time = pd.to_datetime(str(time_value))
        yy, mm, dd, hh, mi = date_time.year, date_time.month, date_time.day, date_time.hour, date_time.minute
        
        df_time_tmp = pd.DataFrame({
            'year': [yy], 'month': [mm], 'day': [dd], 'hour': [hh], 'minute': [mi]
        })
        
        # Convert to binary image based on OLR threshold
        # 1 = deep convection (OLR < threshold), 0 = no deep convection
        image = np.where(ds_time.data < cut, 1, 0).astype(int)
        
        # Compute organization metrics
        if periodic:
            dict_org = run_metrics_periodic(np.copy(image))
        else:
            dict_org = run_metrics(np.copy(image))
        
        # Aggregate results
        df_org_tmp = pd.DataFrame(dict_org, index=[0])
        df_tmp = pd.concat([df_time_tmp.reset_index(drop=True), df_org_tmp.reset_index(drop=True)], axis=1)
        df = pd.concat([df, df_tmp], ignore_index=True)
    
    print(f"✓ Processing complete!")
    return df

## 7. Compute Organization Indices

This may take several minutes depending on the dataset size.

In [ ]:
print(f"Computing organization indices with:")
print(f"  - OLR threshold: {OLR_THRESHOLD} W/m²")
print(f"  - Periodic boundaries: {USE_PERIODIC}")
print(f"  - Using: {'Fixed periodic implementation' if USE_PERIODIC else 'Standard implementation'}")
print()

df_indices = process_organization_indices(
    olr_data,
    cut=OLR_THRESHOLD,
    periodic=USE_PERIODIC
)

# Add time column from original data
df_indices['time'] = olr_data.time.values

print(f"\n✓ Organization indices computed for {len(df_indices)} time steps")
print(f"\nDataFrame columns: {list(df_indices.columns)}")

## 8. Quick Summary Statistics

In [ ]:
print("="*70)
print("SUMMARY STATISTICS")
print("="*70)

summary_metrics = ['number', 'mean_area', 'Iorg', 'SCAI', 'MCAI', 'COP', 'ROME', 'Lorg']

for metric in summary_metrics:
    if metric in df_indices.columns:
        mean_val = df_indices[metric].mean()
        std_val = df_indices[metric].std()
        min_val = df_indices[metric].min()
        max_val = df_indices[metric].max()
        print(f"\n{metric}:")
        print(f"  Mean: {mean_val:.3f}")
        print(f"  Std:  {std_val:.3f}")
        print(f"  Min:  {min_val:.3f}")
        print(f"  Max:  {max_val:.3f}")

print("\n" + "="*70)

## 9. Display First Few Rows

In [ ]:
df_indices.head(10)

## 10. Plot Time Series

### 10.1 Main Organization Indices (Iorg, SCAI, MCAI)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax2 = ax.twinx()

# Plot Iorg on left axis
ax.plot(df_indices['time'], df_indices['Iorg'], color='k', label='Iorg', alpha=0.8, linewidth=1.5)

# Plot SCAI and MCAI on right axis
ax2.plot(df_indices['time'], df_indices['SCAI'], color='orange', label='SCAI', alpha=0.8, linewidth=1.5)
ax2.plot(df_indices['time'], df_indices['MCAI'], color='r', label='MCAI', alpha=0.8, linewidth=1.5)

# Format axes
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Iorg', fontsize=11)
ax.yaxis.label.set_color('k')
ax.set_ylim(0, 1)

ax2.set_ylabel('SCAI / MCAI', fontsize=11)
ax2.yaxis.label.set_color('r')
ax2.set_ylim(-7, 0)

# Legends
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right', frameon=False, fontsize=10)

plt.title(f'{EXPERIMENT_LABEL} - Organization Indices Time Series', fontsize=12)
plt.tight_layout()
plt.show()

### 10.2 Number of Objects and Mean Area

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Number of objects
axes[0].plot(df_indices['time'], df_indices['number'], color='darkblue', alpha=0.7, linewidth=1)
axes[0].set_ylabel('Number of Objects', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Deep Convective Objects Detected', fontsize=11)

# Mean area
axes[1].plot(df_indices['time'], df_indices['mean_area'], color='darkgreen', alpha=0.7, linewidth=1)
axes[1].set_ylabel('Mean Area [grid points]', fontsize=11)
axes[1].set_xlabel('Time', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Mean Object Area', fontsize=11)

plt.suptitle(f'{EXPERIMENT_LABEL} - Object Statistics', fontsize=12, y=0.995)
plt.tight_layout()
plt.show()

### 10.3 All Organization Indices

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, layout='constrained')

# Panel 1: Iorg, OIDRA, COP
ax1_twin = axes[0].twinx()
axes[0].plot(df_indices['time'], df_indices['Iorg'], color='k', label='Iorg', alpha=0.8, linewidth=1.5)
axes[0].plot(df_indices['time'], df_indices['OIDRA'], color='g', label='OIDRA', alpha=0.8, linewidth=1.5)
ax1_twin.plot(df_indices['time'], df_indices['COP'], color='peru', label='COP', alpha=0.8, linewidth=1.5)
axes[0].set_ylabel('Iorg / OIDRA', fontsize=10)
ax1_twin.set_ylabel('COP', fontsize=10, color='peru')
axes[0].set_ylim(-0.1, 1.1)
axes[0].legend(loc='upper left', frameon=False, fontsize=9)
ax1_twin.legend(loc='upper right', frameon=False, fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: SCAI, MCAI
axes[1].plot(df_indices['time'], df_indices['SCAI'], color='orange', label='SCAI', alpha=0.8, linewidth=1.5)
axes[1].plot(df_indices['time'], df_indices['MCAI'], color='r', label='MCAI', alpha=0.8, linewidth=1.5)
axes[1].set_ylabel('SCAI / MCAI', fontsize=10)
axes[1].set_ylim(-5, 0.05)
axes[1].legend(loc='upper left', frameon=False, fontsize=9)
axes[1].grid(True, alpha=0.3)

# Panel 3: ROME, Lorg
ax3_twin = axes[2].twinx()
axes[2].plot(df_indices['time'], df_indices['ROME'], color='plum', label='ROME', alpha=0.8, linewidth=1.5)
ax3_twin.plot(df_indices['time'], df_indices['Lorg'], color='skyblue', label='Lorg', alpha=0.8, linewidth=1.5)
axes[2].set_ylabel('ROME', fontsize=10, color='plum')
ax3_twin.set_ylabel('Lorg', fontsize=10, color='skyblue')
axes[2].set_xlabel('Time', fontsize=11)
axes[2].legend(loc='upper left', frameon=False, fontsize=9)
ax3_twin.legend(loc='upper right', frameon=False, fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'{EXPERIMENT_LABEL} - All Organization Indices', fontsize=12)
plt.show()

### 10.4 Daily Averaged Time Series (Optional)

If your simulation spans multiple days, compute and plot daily averages.

In [ ]:
# Compute daily averages
df_indices['day_of_year'] = pd.to_datetime(df_indices['time']).dt.dayofyear
df_daily = df_indices.select_dtypes(include='number').groupby('day_of_year').mean()

print(f"Daily averaged data: {len(df_daily)} days")

# Plot daily averages
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax2 = ax.twinx()

ax.plot(df_daily.index, df_daily['Iorg'], 'o-', color='k', label='Iorg', alpha=0.8, markersize=4)
ax2.plot(df_daily.index, df_daily['SCAI'], 'o-', color='orange', label='SCAI', alpha=0.8, markersize=4)
ax2.plot(df_daily.index, df_daily['MCAI'], 'o-', color='r', label='MCAI', alpha=0.8, markersize=4)

ax.set_xlabel('Day of Year', fontsize=11)
ax.set_ylabel('Iorg', fontsize=11)
ax.yaxis.label.set_color('k')
ax.set_ylim(0, 1)

ax2.set_ylabel('SCAI / MCAI', fontsize=11)
ax2.yaxis.label.set_color('r')
ax2.set_ylim(-7, 0)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right', frameon=False, fontsize=10)

ax.grid(True, alpha=0.3)
plt.title(f'{EXPERIMENT_LABEL} - Daily Averaged Organization Indices', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Save Results to Pickle File

In [ ]:
# Create output directory if it doesn't exist
output_dir_path = Path(OUTPUT_DIR)
output_dir_path.mkdir(exist_ok=True, parents=True)

# Full output path
output_path = output_dir_path / OUTPUT_FILENAME

# Save to pickle
df_indices.to_pickle(output_path)

print("="*70)
print("RESULTS SAVED")
print("="*70)
print(f"File: {output_path}")
print(f"Size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"Rows: {len(df_indices)}")
print(f"Columns: {len(df_indices.columns)}")
print("\nColumn names:")
for col in df_indices.columns:
    print(f"  - {col}")
print("\n✓ Save complete!")

## 12. Verification: Load Saved File

Quick check to verify the saved file can be loaded correctly.

In [ ]:
# Load the saved file
df_loaded = pd.read_pickle(output_path)

print("Verification: File loaded successfully!")
print(f"Shape: {df_loaded.shape}")
print(f"\nFirst 5 rows:")
display(df_loaded.head())

print(f"\nLast 5 rows:")
display(df_loaded.tail())

## Summary

**What this notebook does:**
1. Loads OLR data from PINACLES or DP-SCREAM simulations
2. Computes convective organization indices using the **fixed periodic boundary implementation**
3. Generates comprehensive time series plots
4. Saves results to pickle file for later analysis

**Output file contains:**
- Time information (year, month, day, hour, minute, time)
- Object statistics (number, area, mean_area)
- Organization indices: Iorg, Lorg, SCAI, MCAI, COP, ABCOP, ROME, OIDRA

**To use this notebook for a different simulation:**
1. Modify the configuration cell (Section 1)
2. Update file paths, variable names, and domain information
3. Run all cells